**Lab5_Ex_5_By_Prajakta Ghorpade SCFP125004**

Exercise 5 — Customer Support Chat Log to Escalation Report Pipeline
Build a 3-step pipeline. Step 1 normalizes a messy support chat log into a structured summary (issue, customer tone, resolution attempted). Step 2 checks the structured summary for missing information needed to escalate the issue (e.g. account ID, error code) and flags any gaps. Step 3 generates a prioritized escalation report for the support lead based on the structured, gap-checked version.

Input: chat_log_text

Output: the structured summary from Step 1, the flagged gaps from Step 2, and the escalation report from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: issue, customer_tone, resolution_attempted)

Step 2 — Template: ChatPromptTemplate (system persona: "escalation-readiness reviewer")  |  Parser: PydanticOutputParser (fields: missing_fields: list, is_ready_to_escalate: bool)

Step 3 — Template: ChatPromptTemplate (system persona: "support operations lead")  |  Parser: StrOutputParser



Install required libraries

In [3]:
%pip install -q langchain langchain-openai pydantic
%pip install -q langchain-groq langchain langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.9 MB/s eta 0:00:00


Step 1 — Import libraries and configure the language model

In [4]:

import os
from typing import List

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

llm.invoke("hi")


AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.064302138, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003975409, 'prompt_tokens_details': None, 'queue_time': 0.004034387, 'total_time': 0.068277547}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16786-90c0-70b1-892e-79badf18f9cc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

Step2=Structured Customer Support Summary.

In [5]:
class SupportSummary(BaseModel):
    issue: str = Field(description="Main issue reported by the customer")
    customer_tone: str = Field(description="Customer's tone or emotion")
    resolution_attempted: str = Field(description="Actions already attempted to resolve the issue")


parser = PydanticOutputParser(pydantic_object=SupportSummary)

summary_prompt = PromptTemplate(
    template="""
Analyze the following customer support chat.

Extract:
1. The main issue
2. The customer's tone
3. The resolution attempts already made

Chat log:
{chat_log_text}

{format_instructions}
""",
    input_variables=["chat_log_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

step1_chain = summary_prompt | llm | parser

In [6]:
chat_log_text = """
Customer: I paid for my order, but it has not arrived.
Agent: I checked the tracking details and asked the delivery team for an update.
Customer: This is frustrating. I have been waiting for five days.
"""

summary = step1_chain.invoke({"chat_log_text": chat_log_text})

print(summary.model_dump())

{'issue': 'Order has not arrived after payment', 'customer_tone': 'Frustrated', 'resolution_attempted': 'Agent checked tracking details and requested an update from the delivery team'}


Define the output structure and prompt

In [7]:
class EscalationCheck(BaseModel):
    missing_fields: List[str] = Field(
        description="List of information missing from the support case"
    )
    is_ready_to_escalate: bool = Field(
        description="Whether the case is ready for escalation"
    )


escalation_parser = PydanticOutputParser(
    pydantic_object=EscalationCheck
)

escalation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an escalation-readiness reviewer. "
        "Check whether the customer support case has enough information "
        "for escalation. Identify missing information and decide readiness. "
        "Do not invent facts.\n{format_instructions}"
    ),
    (
        "human",
        "Customer support summary:\n{summary}\n\n"
        "Original chat log:\n{chat_log_text}"
    )
]).partial(
    format_instructions=escalation_parser.get_format_instructions()
)

step2_chain = escalation_prompt | llm | escalation_parser

In [8]:
escalation_result = step2_chain.invoke({
    "summary": summary.model_dump_json(),
    "chat_log_text": chat_log_text
})

print(escalation_result.model_dump())

{'missing_fields': ['Order ID', 'Customer name', 'Contact information (email/phone)', 'Payment confirmation or transaction ID', 'Shipping address', 'Tracking number', 'Expected delivery date', 'Reference numbers'], 'is_ready_to_escalate': False}


Step 3: Generate the Prioritized Escalation Report.

In [9]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a support operations lead. "
        "Prepare a clear, professional, prioritized escalation report "
        "using only the information provided. "
        "Include the customer issue, customer tone, resolution attempts, "
        "missing information, escalation readiness, and recommended next steps. "
        "Do not invent facts."
    ),
    (
        "human",
        """
Customer Support Summary:
{summary}

Escalation Readiness Check:
{escalation_result}

Original Chat Log:
{chat_log_text}

Generate the prioritized escalation report.
"""
    )
])

step3_chain = report_prompt | llm | StrOutputParser()

In [10]:
escalation_report = step3_chain.invoke({
    "summary": summary.model_dump_json(),
    "escalation_result": escalation_result.model_dump_json(),
    "chat_log_text": chat_log_text
})

print(escalation_report)

**Escalation Report – Order Delivery Issue**

| Item | Details |
|------|---------|
| **Customer Issue** | Order has not arrived after payment. |
| **Customer Tone** | Frustrated (expressed dissatisfaction after five days of waiting). |
| **Resolution Attempts** | • Agent verified tracking details.<br>• Agent requested an update from the delivery team. |
| **Missing Information** | • Order ID<br>• Customer name<br>• Contact information (email/phone)<br>• Payment confirmation or transaction ID<br>• Shipping address<br>• Tracking number<br>• Expected delivery date<br>• Reference numbers (e.g., internal ticket, delivery reference) |
| **Escalation Readiness** | **Not ready** – key data required for a full escalation is absent. |
| **Priority** | **High** – customer has been waiting 5 days; frustration level is elevated. |

---

### Recommended Next Steps

1. **Collect Missing Data (Immediate)**
   - Reach out to the customer via the last known contact method (email or phone) to request:
 

In [11]:
print("========== STEP 1: CUSTOMER SUPPORT SUMMARY ==========")
print(summary.model_dump_json(indent=2))

print("\n========== STEP 2: ESCALATION READINESS CHECK ==========")
print(escalation_result.model_dump_json(indent=2))

print("\n========== STEP 3: PRIORITIZED ESCALATION REPORT ==========")
print(escalation_report)

========== STEP 1: CUSTOMER SUPPORT SUMMARY ==========
{
  "issue": "Order has not arrived after payment",
  "customer_tone": "Frustrated",
  "resolution_attempted": "Agent checked tracking details and requested an update from the delivery team"
}

========== STEP 2: ESCALATION READINESS CHECK ==========
{
  "missing_fields": [
    "Order ID",
    "Customer name",
    "Contact information (email/phone)",
    "Payment confirmation or transaction ID",
    "Shipping address",
    "Tracking number",
    "Expected delivery date",
    "Reference numbers"
  ],
  "is_ready_to_escalate": false
}

========== STEP 3: PRIORITIZED ESCALATION REPORT ==========
**Escalation Report – Order Delivery Issue**

| Item | Details |
|------|---------|
| **Customer Issue** | Order has not arrived after payment. |
| **Customer Tone** | Frustrated (expressed dissatisfaction after five days of waiting). |
| **Resolution Attempts** | • Agent verified tracking details.<br>• Agent requested an update from the deliv